In [1]:
import os

In [2]:
%pwd

'd:\\User\\Akhilesh\\MLOps _Projects\\datascienceproject\\research'

In [3]:
os.chdir("../")
%pwd

'd:\\User\\Akhilesh\\MLOps _Projects\\datascienceproject'

In [ ]:
from dataclasses import dataclass
from pathlib import Path

@dataclass
class DataIngestionConfig:
    root_dir: Path
    source_URL: str
    local_data_file: Path
    unzip_dir: Path

In [6]:
from src.datascience.constants import *
from src.datascience.utils.common import read_yaml, create_directories

In [ ]:
class ConfigurationManager:
    def __init__(self, 
                 config_file_path = CONFIG_FILE_PATH,
                 params_file_path = PARAMS_FILE_PATH,
                 schema_file_path = SCHEMA_FILE_PATH):
        self.config = read_yaml(config_file_path)
        self.params = read_yaml(params_file_path)
        self.schema = read_yaml(schema_file_path)

        create_directories([self.config.artifacts_root])

    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion
        create_directories([config.root_dir])

        data_ingestion_config = DataIngestionConfig(
            root_dir = config.root_dir,
            source_URL = config.source_URL,
            local_data_file = config.local_data_file,
            unzip_dir = config.unzip_dir
        )

        return data_ingestion_config


In [14]:
import os
import urllib.request as request
from src.datascience import logger
import zipfile

In [ ]:
## component 1: data ingestion

class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config

    # downloading the zip file 
    def download_file(self):
        if not os.path.exists(self.config.local_data_file):
            filename, headers = request.urlretrieve(
                url = self.config.source_URL,
                filename = self.config.local_data_file
            )
            logger.info(f"{filename} downloaded! with following info: \n{headers}")
        else:
            logger.info(f"File already exists")

    def extract_zip_file(self):
        """
        Unzip the downloaded file into the unzip_dir
        """
        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path, exist_ok=True)
        with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_ref:
            zip_ref.extractall(unzip_path)

In [ ]:
try:
    config = ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion_config()
    data_ingestion = DataIngestion(config=data_ingestion_config)
    data_ingestion.download_file()
    data_ingestion.extract_zip_file()

except Exception as e:
    raise e

[2026-03-11 15:36:47,576]: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-03-11 15:36:47,578]: INFO: common: yaml file: params.yaml loaded successfully]
[2026-03-11 15:36:47,580]: INFO: common: yaml file: schema.yaml loaded successfully]
[2026-03-11 15:36:47,582]: INFO: common: created directory at: artifacts]
[2026-03-11 15:36:47,584]: INFO: common: created directory at: artifacts/data_injestion]
[2026-03-11 15:36:48,689]: INFO: 4029457542: artifacts/data_injestion/data.zip downloaded! with following info: 
Connection: close
Content-Length: 23329
Cache-Control: max-age=300
Content-Security-Policy: default-src 'none'; style-src 'unsafe-inline'; sandbox
Content-Type: application/zip
ETag: "c69888a4ae59bc5a893392785a938ccd4937981c06ba8a9d6a21aa52b4ab5b6e"
Strict-Transport-Security: max-age=31536000
X-Content-Type-Options: nosniff
X-Frame-Options: deny
X-XSS-Protection: 1; mode=block
X-GitHub-Request-Id: B4B0:F3BDF:AAB75:1D5005:69B13EB3
Accept-Ranges: bytes
Date: W